# Hierarchical method map

This notebook inspects the M5 hierarchy used by `m5.hierarchy` and maps reconciliation methods to the places they can improve the forecast. It is intentionally light: no full CV is run here.

In [1]:
from pathlib import Path

import pandas as pd

from m5.config import SETTINGS
from m5.hierarchy import M5_LEVELS_SPEC, build_hierarchy
from m5.recipes import Recipe

long_path = SETTINGS.processed_dir / "long.parquet"
recipe_path = Path("configs/m5/hier_experiments.yaml")
long_path, recipe_path

(PosixPath('data/processed/long.parquet'),
 PosixPath('configs/m5/hier_experiments.yaml'))

Load a capped `long.parquet` first. For local exploration, build it with:

```bash
M5_N_SERIES=500 M5_LAST_N_DAYS=200 make prep
```

In [2]:
df = pd.read_parquet(long_path)
df["ds"] = pd.to_datetime(df["ds"])
df.shape, df["unique_id"].nunique(), df["ds"].min(), df["ds"].max()

((1004671, 18),
 5000,
 Timestamp('2015-11-04 00:00:00'),
 Timestamp('2016-05-22 00:00:00'))

In [3]:
hier = build_hierarchy(df)
level_summary = pd.DataFrame(
    {
        "level": ["/".join(level) for level in M5_LEVELS_SPEC],
        "n_series": [len(hier.tags["/".join(level)]) for level in M5_LEVELS_SPEC],
    }
)
level_summary

,level,n_series
0,Total,1
1,Total/state_id,3
2,Total/state_id/store_id,10
3,Total/cat_id,3
4,Total/cat_id/dept_id,7
5,Total/state_id/cat_id,9
6,Total/state_id/cat_id/dept_id,21
7,Total/state_id/store_id/cat_id,30
8,Total/state_id/store_id/cat_id/dept_id,70
9,Total/cat_id/dept_id/item_id,2538


In [4]:
recipe = Recipe.from_yaml(recipe_path)
pd.DataFrame({"reconciler": recipe.model.reconcilers})

,reconciler
0,BU
1,MinT_OLS
2,MinT_WLS_struct
3,MinT_WLS_var
4,MinT_shrink
5,ERM_closed
6,ERM_reg_bu


## Where each method can help

| Method | Likely improvement | Main risk |
|---|---|---|
| BottomUp | Strong bottom-level item-store forecasts stay untouched; upper levels become coherent. | Does not repair noisy sparse bottom forecasts. |
| MinTrace OLS | Cheap coherent adjustment across all 12 levels. | Treats all coherency errors similarly. |
| MinTrace WLS structural | Gives larger aggregates different variance weight based on hierarchy structure. | Still ignores residual variance. |
| MinTrace WLS variance | Uses in-sample residual variance, often useful when stores/items have unequal noise. | Needs fitted residuals; unstable with very short histories. |
| MinTrace shrinkage | Uses a regularized residual covariance estimate; often the strongest all-level reconciliation candidate. | Heavier on full M5. Use capped CV before full runs. |
| ERM closed | Learns reconciliation directly from fitted-value errors, useful when unbiasedness assumptions are weak. | Can overfit with limited fitted history. |
| ERM reg_bu | Regularizes ERM toward BottomUp, useful for small capped samples. | May under-adjust if the BottomUp prior is too strong. |

`TopDown` and `MiddleOut` are excluded because Nixtla requires a strictly hierarchical tree for those methods. M5 is grouped: product and geography aggregations overlap, so some nodes have multiple parents.